# fintrack — 03 Category Tuning
**Purpose**: Review uncategorised ('Other') transactions and update keywords/merchant  
overrides in the database — without redeploying the API.  
**Version**: 1.0 — 2026-03-24  

**Workflow**:
1. Run Section 1 to see all 'Other' transactions sorted by amount
2. Identify the merchant → correct category mapping
3. Run Section 2 to add keywords to the categories table in the DB
4. Run Section 3 to re-categorise all affected transactions
5. Verify the 'Other' count dropped

> Note: Changes made here update the DB directly. Re-import is NOT needed — 
> Section 3 re-categorises existing transactions in place.

In [1]:
import os
import httpx
import psycopg2
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

API_BASE  = f"http://{os.getenv('API_HOST','192.168.1.170')}:{os.getenv('API_PORT','8000')}"
EMAIL     = os.getenv('FINTRACK_EMAIL', 'mahajani6630@gmail.com')
PASSWORD  = os.getenv('FINTRACK_PASSWORD', 'Akamo@6630')
DB_HOST   = os.getenv('DB_HOST', '192.168.1.169')
DB_PORT   = int(os.getenv('DB_PORT', '5432'))
DB_NAME   = os.getenv('DB_NAME', 'fintrack')
DB_USER   = os.getenv('DB_USER', 'fintrack')
DB_PASS   = os.getenv('DB_PASSWORD', '')

# Authenticate
r = httpx.post(f'{API_BASE}/api/v1/auth/login',
               json={'email': EMAIL, 'password': PASSWORD}, timeout=10)
assert r.status_code == 200, f'Login failed: {r.text}'
TOKEN   = r.json()['access_token']
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
USER_ID = r.json()['user_id']
print(f'API: authenticated as {EMAIL}')

# DB connection for direct updates
conn = psycopg2.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME,
                         user=DB_USER, password=DB_PASS)
conn.autocommit = False
print(f'DB:  connected to {DB_HOST}/{DB_NAME}')

API: authenticated as mahajani6630@gmail.com
DB:  connected to 192.168.1.169/fintrack


## Section 1 — Review 'Other' transactions

In [2]:
# Fetch all 'Other' transactions via API (descriptions decrypted)
params = {'password': PASSWORD, 'category': 'Other', 'page_size': 200}
r = httpx.get(f'{API_BASE}/api/v1/transactions', headers=HEADERS,
              params=params, timeout=15)
data = r.json()

df_other = pd.DataFrame(data['items'])
df_other['amount'] = df_other['amount'].astype(float)
df_other['date']   = pd.to_datetime(df_other['date'])

print(f"Uncategorised transactions: {data['total']}")
print(f"Total amount in Other:     ${df_other['amount'].sum():,.2f}")
print()

# Show sorted by amount desc for easy prioritisation
df_other[['date','description','amount','source_file']] \
    .sort_values('amount', ascending=False) \
    .reset_index(drop=True)

KeyError: 'items'

In [3]:
# Show unique descriptions sorted by total spend — helps identify patterns
df_other.groupby('description')['amount'].agg(['sum','count']) \
    .sort_values('sum', ascending=False) \
    .rename(columns={'sum':'total','count':'txns'}) \
    .reset_index()

NameError: name 'df_other' is not defined

## Section 2 — Add keywords to categories

Edit the `NEW_KEYWORDS` dict below then run the cell.  
Format: `'Category Name': ['keyword1', 'keyword2', ...]`  
Keywords are appended to existing ones — existing keywords are preserved.

In [4]:
NEW_CATEGORIES = [
    # (name, subcategory, is_essential, color_code, sort_order)
    ('Transport',        'Car Maintenance', True,  '#00695C', 24),
    ('Home Improvement', 'Home Maintenance',True,  '#E65100', 51),
    ('Fees & Interest',  'Tax',            True,  '#B71C1C', 142),
]

cur = conn.cursor()
for name, subcat, essential, color, sort in NEW_CATEGORIES:
    cur.execute("""
        INSERT INTO categories 
            (user_id, name, subcategory, is_essential, color_code, keywords, sort_order)
        VALUES (%s, %s, %s, %s, %s, %s::text[], %s)
        ON CONFLICT (user_id, name, subcategory) DO NOTHING
    """, (USER_ID, name, subcat, essential, color, [], sort))
    print(f"Added: {name} / {subcat}")

conn.commit()
print("Done")

Added: Transport / Car Maintenance
Added: Home Improvement / Home Maintenance
Added: Fees & Interest / Tax
Done


In [5]:
# ── EDIT THIS DICT ────────────────────────────────────────────────────────────
# Add your new keyword mappings here.
# Key = exact category name (must match categories table)
# Value = list of new keywords to add

NEW_KEYWORDS = {
    # Examples — replace with your actual findings from Section 1:
    # 'Groceries':      ['central market', 'suvidha'],
    # 'Dining':         ['flippin pizza', 'get n geaux'],
    # 'Transport':      ['citgo'],
    # 'Entertainment':  ['ruby falls', 'incline'],
    # 'Pet Care':       ['bernas canine', 'livepawsiti'],
        "Groceries":     ["grocery", "supermarket", "whole foods", "trader joe", "safeway",
                          "kroger", "publix", "tom thumb", "aldi", "sprouts", "heb ",
                          "food lion", "wegmans", "stop & shop", "giant", "meijer",
                          "winn-dixie", "market", "fresh market", "indian plaza", "indifresh", "patel brothers",
                          "sp soul foods", "cherians", "wal-mart", "bombay bazaar",
                          "spices hut indian grocer"],
          "Dining":        ["restaurant", "cafe", "coffee", "starbucks", "mcdonald", "subway",
                          "pizza", "sushi", "doordash", "grubhub", "uber eats", "dining",
                          "bar ", "tavern", "grill", "bistro", "diner", "chick-fil",
                          "chipotle", "taco bell", "wendy", "burger king", "panera",
                          "olive garden", "applebee", "ihop", "denny", "tst* indi fresh", "pf changs",
                          "sri krishna vilas", "tgi friday's", "schlotzsky's", "thai basil"
                          "the curry kitchen", "kakatiya indian kitch", "eclipse di luna",
                          "tst* tasty delights", "flames mediterranean", "cinco mexican cantina",
                          "tst*indi fresh", "ramirez mexican restauran", "the curry kitchen",
                          "papa johns", "village italian", "mh15", "cantina", "las palmas mexican",
                          "kwality ice creams atlan", "rest bella pasta", 
                          "italian ice", "taste of india", "thai basil kitchen",
                          "whataburger", "chuy's"],
        "Transport":     ["uber", "lyft", "taxi", "transit", "metro", "train", "bus ",
                          "parking", "toll"],
        "Gas":           ["qt", "gas station", "shell", "exxon", "bp ",
                          "chevron", "sunoco", "marathon", "valero", "murphy",
                          "airline", "flight", "amtrak", "marta", "circle k", "racetrac", "raceway",
                          "citgo", "circle m"],
        "Shopping":      ["amazon", "walmart", "target", "ebay", "etsy", "best buy",
                          "apple store", "nike", "zara", "h&m", "gap", "nordstrom", "macy",
                          "tj maxx", "marshalls", "ross ", "homegoods", "bed bath", "wm supercenter",
                          "gandhi appl 4029357733 ca", "ppusind pernia 7847848484 sg", "instacart",
                          "duty free", "airwalxuk*curvedreamparis", "murad rupani"],
        "Health":        ["pharmacy", "cvs", "walgreens", "rite aid", "dentist",
                          "medical", "health", "gym", "fitness", "yoga",
                          "urgent care", "optometrist", "vision", "questdiagno", "thomas eye group"],
        "Entertainment": ["netflix", "spotify", "hulu", "disney", "amazon prime", "apple tv",
                          "cinema", "movie", "theater", "concert", "ticketmaster", "steam",
                          "gaming", "playstation", "xbox", "sling.com",
                          "world of coca cola", "roku", "ruby falls",
                          "incline www.ridetheinga", "tm* maheshkale-rezo"],
        "Hotel":         ["hotel", "motel", "inn ", "resort", "lodge", "suites",
                          "airbnb", "vrbo", "marriott", "hilton", "hyatt", "hampton inn",
                          "holiday inn", "best western", "sheraton", "westin", "ritz",
                          "expedia", "booking.com", "laquinta"],
        "Car Rental":    ["car rental", "hertz", "enterprise",
                          "avis ", "budget rent", "national car"],
        "Air fare":      ["delta air lines", "cl *chase travel"],
        "Travel":        ["railninja", "sintra", "porto", "portugal"],
        "Car Maintain":  ["supershine"],
        "Home Improv":   ["the home depot", "floor and decor", "lowe's", "builders surplus",
                          "legacy house of windows"],
        "Home Maintain": ["mr hvac heating  coocanton"],
        "Utilities":     ["electric", "water ", "internet", "cable", "phone",
                          "at&t", "verizon", "t-mobile", "comcast", "spectrum",
                          "duke energy", "dominion energy", "xfinity", "legacy disposal",
                          "py *urbanex atlanta", "red oak sanitation", "ring standard plan",
                          "car wash", "fc water&sewer"],
        "Insurance":     ["insurance", "geico", "allstate", "progressive", "state farm",
                          "farmers ", "liberty mutual", "nationwide"],
        "Education":     ["tuition", "university", "college", "coursera", "udemy", "book", "gsu newton",
                          "awl*pearson education", "pluralsight"],
        "Personal Care": ["salon", "spa", "great clips", "haircut", "barber", "beauty", "nail",
                          "geek gorgeous"],
        "Pet Care":      ["urgent vet", "rover.com", "camp bow wow", "big creek animal",
                          "animal", "vine animal hospital", "animaldoctor", "gdp*bernas canine cottag duluth ga"],
        "Membership":    ["costco *annual renewal", "annual membership fee", "renewal membership fee"],
                          "Late Fee, Int": ["late fee", "interest charge on purchases"],
        "Tax Consult"  : ["in *douglas segars cpa, l770-8262982 ga"],
        "Traffic Tkt"  : ["arlington municipal co arlington tx"],
        "GA Tax"       : ["forsyth mvd"],
        "Internet"     : ["dnh*godaddy#"],

}
# ── END EDIT ──────────────────────────────────────────────────────────────────

if not NEW_KEYWORDS:
    print('Nothing to add — populate NEW_KEYWORDS dict above')
else:
    cur = conn.cursor()
    for cat_name, new_kws in NEW_KEYWORDS.items():
        # Fetch current keywords
        cur.execute("""
            SELECT id, keywords FROM categories
            WHERE user_id = %s AND name = %s
            LIMIT 1
        """, (USER_ID, cat_name))
        row = cur.fetchone()
        if not row:
            print(f"  WARNING: Category '{cat_name}' not found — skipping")
            continue
        cat_id, existing_kws = row
        existing_kws = existing_kws or []
        # Merge, avoiding duplicates
        merged = list(dict.fromkeys(existing_kws + [k.lower() for k in new_kws]))
        cur.execute("""
            UPDATE categories SET keywords = %s
            WHERE id = %s
        """, (merged, cat_id))
        added = [k for k in new_kws if k.lower() not in existing_kws]
        print(f"  {cat_name}: added {len(added)} keyword(s): {added}")
    conn.commit()
    print('\nKeywords saved to database.')

  Groceries: added 27 keyword(s): ['grocery', 'supermarket', 'whole foods', 'trader joe', 'safeway', 'kroger', 'publix', 'tom thumb', 'aldi', 'sprouts', 'heb ', 'food lion', 'wegmans', 'stop & shop', 'giant', 'meijer', 'winn-dixie', 'market', 'fresh market', 'indian plaza', 'indifresh', 'patel brothers', 'sp soul foods', 'cherians', 'wal-mart', 'bombay bazaar', 'spices hut indian grocer']
  Dining: added 50 keyword(s): ['restaurant', 'mcdonald', 'subway', 'pizza', 'sushi', 'doordash', 'grubhub', 'uber eats', 'dining', 'bar ', 'tavern', 'grill', 'bistro', 'diner', 'chick-fil', 'chipotle', 'taco bell', 'wendy', 'burger king', 'panera', 'olive garden', 'applebee', 'ihop', 'denny', 'tst* indi fresh', 'pf changs', 'sri krishna vilas', "tgi friday's", "schlotzsky's", 'thai basilthe curry kitchen', 'kakatiya indian kitch', 'eclipse di luna', 'tst* tasty delights', 'flames mediterranean', 'cinco mexican cantina', 'tst*indi fresh', 'ramirez mexican restauran', 'the curry kitchen', 'papa johns',

## Section 3 — Re-categorise existing 'Other' transactions

This updates the `category_name` and `subcategory` columns for transactions  
that are currently 'Other' but now match a keyword. No re-import needed.

In [6]:
# Load current categories with keywords from DB
cur = conn.cursor()
cur.execute("""
    SELECT name, subcategory, is_essential, keywords
    FROM categories
    WHERE user_id = %s AND name != 'Other'
    ORDER BY sort_order
""", (USER_ID,))
categories = cur.fetchall()
print(f'Loaded {len(categories)} categories')

# Build keyword → (category, subcategory, is_essential) lookup
# Note: longer/more specific keywords should win over shorter ones
keyword_map = {}  # keyword -> (cat, subcat, essential)
for name, subcat, essential, keywords in categories:
    for kw in (keywords or []):
        if kw and kw not in keyword_map:
            keyword_map[kw.lower()] = (name, subcat or name, essential)

def infer(description):
    desc_lower = description.lower()
    # Check longer keywords first (more specific)
    for kw in sorted(keyword_map.keys(), key=len, reverse=True):
        if kw in desc_lower:
            return keyword_map[kw]
    return None

print('Keyword lookup ready')

Loaded 43 categories
Keyword lookup ready


In [7]:
# Fetch all 'Other' transactions from DB (we need plaintext descriptions
# which we already have from the API call in Section 1)
if df_other.empty:
    print('No Other transactions to process')
else:
    updated = 0
    still_other = []
    preview = []

    for _, row in df_other.iterrows():
        result = infer(row['description'])
        if result:
            cat, subcat, essential = result
            preview.append({'description': row['description'],
                            'amount': row['amount'],
                            'new_category': cat,
                            'new_subcategory': subcat})
        else:
            still_other.append(row['description'])

    print(f'Will re-categorise: {len(preview)} transactions')
    print(f'Still Other:        {len(still_other)} transactions')
    print()
    pd.DataFrame(preview)

NameError: name 'df_other' is not defined

In [8]:
# ── CONFIRM BEFORE RUNNING ────────────────────────────────────────────────────
# Review the preview above, then set CONFIRM = True to apply changes
CONFIRM = False
# ─────────────────────────────────────────────────────────────────────────────

if not CONFIRM:
    print('Set CONFIRM = True to apply re-categorisation')
else:
    # We need transaction IDs from the DB to update them
    # Fetch Other transactions with their IDs directly from DB
    cur.execute("""
        SELECT t.id, a.provider
        FROM transactions t
        JOIN accounts a ON a.id = t.account_id
        WHERE t.user_id = %s AND t.category_name = 'Other'
    """, (USER_ID,))
    db_others = {str(r[0]) for r in cur.fetchall()}

    updated = 0
    for _, row in df_other.iterrows():
        result = infer(row['description'])
        if result and row['id'] in db_others:
            cat, subcat, essential = result
            cur.execute("""
                UPDATE transactions
                SET category_name = %s,
                    subcategory   = %s,
                    is_essential  = %s
                WHERE id = %s::UUID
            """, (cat, subcat, essential, row['id']))
            updated += 1

    conn.commit()
    print(f'Updated {updated} transactions')
    print('Re-run Section 1 to see the new Other count')

Set CONFIRM = True to apply re-categorisation


## Section 4 — Show current category totals

In [9]:
r = httpx.get(f'{API_BASE}/api/v1/analytics/category-summary',
              headers=HEADERS, params={'year': 2025}, timeout=15)
data = r.json()
print(f"Grand Total: ${data['grand_total']:,.2f}")
print(f"Other:       ${next((c['total'] for c in data['categories'] if c['category']=='Other'), 0):,.2f}")
print()
df = pd.DataFrame(data['categories'])
df[['category','subcategory','total','txn_count','pct']].style \
    .background_gradient(subset=['total'], cmap='YlOrRd') \
    .format({'total': '${:,.2f}', 'pct': '{:.1f}%'})

Grand Total: $53,232.72
Other:       $4,586.37



,category,subcategory,total,txn_count,pct
0,Air Travel,Air Travel,"$12,727.67",18,23.9%
1,Home Improvement,Home Improvement,"$8,296.46",28,15.6%
2,Other,Other,"$4,586.37",81,8.6%
3,Shopping,General Retail,"$3,355.59",20,6.3%
4,Groceries,Ethnic Grocery,"$3,089.16",83,5.8%
5,Groceries,Warehouse Club - Food,"$2,759.42",40,5.2%
6,Groceries,Grocery Store,"$2,424.12",102,4.5%
7,Health,Medical & Dental,"$1,650.88",17,3.1%
8,Transport,Fuel,"$1,573.11",44,3.0%
9,Hotel,Hotel,"$1,496.23",8,2.8%
